# Fix the overthinker · Colab version (L02 hands-on)

**First:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells from top to bottom (Shift+Enter).

The whole experiment runs here, from scratch, on Google's GPU. Nothing is installed on your own computer.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers accelerate modelscope flash-linear-attention

## The scripts and the questions
The next four cells write the two scripts and two question sets the experiment uses. They are the same files as in the `weight-space-programming` folder on GitHub.

In [ ]:
%%writefile overthinker.py
#!/usr/bin/env python3
"""Fix an overthinking model by editing its weights, no training.

A small, hands-on version of the Qwen fix in "Sparks of Weight-Space Programming"
(Omar Khattab's group, MIT CSAIL, 2 Nov 2026). Four steps:

  observe    run the model on practice problems; count answers that run out of tokens
  direction  average the residual stream of looped vs finished answers; the difference is u
  patch      remove u from the top-k columns of one layer's MLP output matrix; save the model
  observe    run the patched model on the same problems; count again

Get the model first (ModelScope, no Hugging Face account needed):
  modelscope download --model Qwen/Qwen3.5-0.8B --local_dir ./qwen3.5-0.8b

Usage (--model is the local folder, or any Hugging Face id):
  python overthinker.py observe   --model ./qwen3.5-0.8b
  python overthinker.py direction --model ./qwen3.5-0.8b --layer 15
  python overthinker.py patch     --model ./qwen3.5-0.8b --layer 15 --top 64
  python overthinker.py observe   --model ./qwen-patched
"""
import argparse, json, pathlib, re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

HERE = pathlib.Path(__file__).parent
RUNS = HERE / "runs"; RUNS.mkdir(exist_ok=True)


def device():
    if torch.backends.mps.is_available(): return "mps"
    if torch.cuda.is_available(): return "cuda"
    return "cpu"


def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    except (ValueError, KeyError):
        # Qwen3.5 is listed as image-text-to-text; load the full model and use only its text path
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(name, dtype=torch.float32)
    return tok, model.to(device()).eval()


def layers_of(model):
    """The list of decoder layers (transformer blocks), whatever the architecture calls it."""
    for path in ("model.layers", "model.language_model.layers", "transformer.h", "model.decoder.layers"):
        obj = model
        try:
            for part in path.split("."): obj = getattr(obj, part)
            return obj
        except AttributeError:
            continue
    raise SystemExit("Could not find the decoder layers. Run: python -c \"from transformers import AutoModelForCausalLM as M; print(M.from_pretrained('<model>'))\"")


def mlp_out(layer):
    """The MLP's output matrix (Linear: intermediate -> hidden). Its columns are what the MLP can add to the residual stream."""
    mlp = getattr(layer, "mlp", None) or getattr(layer, "feed_forward", None)
    for name in ("down_proj", "c_proj", "fc2", "w2"):
        if mlp is not None and hasattr(mlp, name): return getattr(mlp, name)
    raise SystemExit(f"Could not find the MLP output matrix in this layer:\n{layer}")


def prompts(tok, n, file="problems.json"):
    qs = json.load(open(HERE / file))[:n]
    out = []
    for q in qs:
        msgs = [{"role": "user", "content": q + "\nPut the final answer in \\boxed{}."}]
        try:
            out.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True))
        except TypeError:
            out.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    return out


def observe(a):
    tok, model = load(a.model)
    ps = prompts(tok, a.n)
    rows, t0 = [], time.time()
    for i in range(0, len(ps), a.batch):
        enc = tok(ps[i:i + a.batch], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=a.max_tokens, do_sample=True, temperature=0.6, top_p=0.95, top_k=20,
                                 pad_token_id=tok.pad_token_id)
        for j, seq in enumerate(gen):
            new = seq[enc["input_ids"].shape[1]:]
            ended = bool((new == tok.eos_token_id).any() or (new == tok.pad_token_id).any())
            rows.append({"prompt": ps[i + j], "output": tok.decode(new, skip_special_tokens=True), "ran_out": not ended})
        print(f"  {len(rows)}/{len(ps)} done · {time.time() - t0:.0f}s")
    tag = pathlib.Path(a.model).name
    json.dump(rows, open(RUNS / f"observe-{tag}.json", "w"), indent=1)
    report(rows, tag, a.max_tokens)


def report(rows, tag, limit):
    for r in rows:
        w = r["output"].split(); g = [tuple(w[i:i + 6]) for i in range(len(w) - 6)]
        r["repeat"] = round(1 - len(set(g)) / max(1, len(g)), 3)      # share of 6-word phrases it has already said
        r["answered"] = bool(re.search(r"\\boxed\{[^{}]*\d[^{}]*\}", r["output"]))  # a boxed answer with a number in it
    print(f"\n{tag}: {sum(r['ran_out'] for r in rows)} / {len(rows)} answers ran out of tokens (limit {limit})")
    print(f"  gave a boxed answer: {sum(r['answered'] for r in rows)} / {len(rows)}")
    print(f"  repeating itself (over 20% repeated phrases): {sum(r['repeat'] > 0.2 for r in rows)} / {len(rows)}")


def score(a):
    tag = pathlib.Path(a.model).name
    report(json.load(open(RUNS / f"observe-{tag}.json")), tag, "as run")
    print(f"saved runs/observe-{tag}.json")


def residual_after(model, enc, layer):
    """Run the model once and return the residual stream right after `layer` (tokens x hidden).
    Uses a hook and skips the vocabulary logits, so an 8,000-token answer fits in GPU memory."""
    got = {}
    def grab(mod, inp, out): got["h"] = (out[0] if isinstance(out, tuple) else out).detach()
    hook = layers_of(model)[layer].register_forward_hook(grab)
    try:
        with torch.no_grad():
            try: model(**enc, logits_to_keep=1)
            except TypeError: model(**enc)
    finally:
        hook.remove()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return got["h"][0]


def loop_start(ids, n=8, times=3):
    """Index of the first token where some n-token phrase appears for the `times`-th time (the loop has begun)."""
    seen = {}
    for i in range(len(ids) - n):
        g = tuple(ids[i:i + n]); seen[g] = seen.get(g, 0) + 1
        if seen[g] == times: return i
    return len(ids)


def direction(a):
    """u = average residual stream while looping, minus a 'not looping' average.
    contrast 'finished': looped answers vs answers that finished (needs both kinds).
    contrast 'early-late': within each looped answer, its last tokens (going round in circles)
                           vs its first tokens of thinking (still reasoning). Works when every answer looped."""
    tok, model = load(a.model)
    rows = json.load(open(RUNS / f"observe-{pathlib.Path(a.model).name}.json"))
    n_loop = sum(r["ran_out"] for r in rows); n_fin = len(rows) - n_loop
    mode = a.contrast
    if mode == "auto": mode = "finished" if n_loop and n_fin else "early-late"
    if mode == "finished" and not (n_loop and n_fin):
        raise SystemExit("Need at least one looped and one finished answer for --contrast finished. Use --contrast early-late.")
    if mode == "early-late" and not n_loop:
        raise SystemExit("No answer ran out of tokens, so there is no loop to study. Lower --max-tokens and observe again.")
    pos, neg = [], []
    for r in rows:
        if mode == "early-late" and not r["ran_out"]: continue
        p_len = tok(r["prompt"], return_tensors="pt")["input_ids"].shape[1]
        enc = tok(r["prompt"] + r["output"], return_tensors="pt").to(model.device)
        h = residual_after(model, enc, a.layer)    # residual stream after layer `layer`, one vector per token
        if mode == "finished":
            (pos if r["ran_out"] else neg).append(h[-a.last:].mean(0).float().cpu())
        else:
            ans = h[p_len:]
            start = loop_start(enc["input_ids"][0, p_len:].tolist())   # first token where the answer starts repeating itself
            k = max(8, min(a.last, start))
            neg.append(ans[:k].mean(0).float().cpu())   # before the loop: still reasoning
            pos.append(ans[-a.last:].mean(0).float().cpu())  # the end: going round in circles
    u = torch.stack(pos).mean(0) - torch.stack(neg).mean(0)
    u = u / u.norm()
    torch.save({"u": u, "layer": a.layer, "contrast": mode, "pos": len(pos), "neg": len(neg)}, RUNS / "direction.pt")
    if mode == "early-late": print("  (start = tokens before each answer begins repeating itself)")
    what = (f"{len(pos)} looped vs {len(neg)} finished answers" if mode == "finished"
            else f"the last tokens vs the first tokens of {len(pos)} looped answers")
    print(f"stuck direction: {u.numel()} dims (the residual stream width), from {what}, after layer {a.layer}")
    print("saved runs/direction.pt")


def writers(layer, hidden, which):
    """Matrices in one layer that write into the residual stream.
    which='mlp': only the MLP output matrix. which='all': every Linear whose output is the residual width
    (the MLP output, and the attention or linear-attention output projection)."""
    if which == "mlp":
        return [("mlp", mlp_out(layer))]
    out_names = ("o_proj", "out_proj", "down_proj", "c_proj", "fc2", "w2")   # output projections only, never input ones
    return [(n, m) for n, m in layer.named_modules()
            if isinstance(m, torch.nn.Linear) and m.out_features == hidden and n.split(".")[-1] in out_names]


def patch(a):
    tok, model = load(a.model)
    d = torch.load(RUNS / "direction.pt")
    u = d["u"].to(model.device)
    hidden = u.numel()
    total = 0
    if a.top <= 0: a.top = 10**9
    layers = layers_of(model)
    picks = range(len(layers)) if a.layer == ["all"] else [int(x) for x in a.layer]
    for L in picks:
        for name, lin in writers(layers[L], hidden, a.writers):
            W = lin.weight.data                                  # shape [hidden, in]; each column writes into the residual stream
            scores = (u @ W).abs()                               # how strongly each column writes along u
            idx = scores.topk(min(a.top, W.shape[1])).indices    # --top 0 means every column
            cols = W[:, idx]
            W[:, idx] = cols - torch.outer(u, u @ cols)          # remove the u part of those columns, keep the rest
            total += cols.numel()
            print(f"layer {L} {name}: edited {len(idx)} of {W.shape[1]} columns ({cols.numel():,} weights)")
    n = sum(p.numel() for p in model.parameters())
    out = HERE / a.out
    model.save_pretrained(out); tok.save_pretrained(out)
    print(f"edited {total:,} of {n:,} weights ({100 * total / n:.4f}%) · saved {out}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    o = sp.add_parser("observe"); o.add_argument("--model", required=True); o.add_argument("--n", type=int, default=32)
    o.add_argument("--max-tokens", type=int, default=1024); o.add_argument("--batch", type=int, default=8)
    d = sp.add_parser("direction"); d.add_argument("--model", required=True); d.add_argument("--layer", type=int, default=15)
    d.add_argument("--last", type=int, default=256)
    d.add_argument("--contrast", choices=["auto", "finished", "early-late"], default="auto")
    p = sp.add_parser("patch"); p.add_argument("--model", required=True); p.add_argument("--layer", nargs="+", default=["15"], help="layer numbers, or: all")
    p.add_argument("--writers", choices=["mlp", "all"], default="mlp", help="mlp: MLP output only; all: every matrix that writes to the residual stream")
    p.add_argument("--top", type=int, default=64); p.add_argument("--out", default="qwen-patched")
    s = sp.add_parser("score"); s.add_argument("--model", required=True)
    a = ap.parse_args()
    {"observe": observe, "direction": direction, "patch": patch, "score": score}[a.cmd](a)


In [ ]:
%%writefile problems.json
[
"A shop sells pens at 3 for 45 rupees. How much do 7 pens cost?",
"A train leaves at 9:40 and arrives at 13:05. How many minutes is the journey?",
"Riya has 128 stickers. She gives a quarter to her brother and 17 to a friend. How many are left?",
"A rectangle has a perimeter of 54 cm and a length of 15 cm. What is its area?",
"If 5 machines make 5 parts in 5 minutes, how many minutes do 100 machines need to make 100 parts?",
"The sum of three consecutive even numbers is 132. What is the largest?",
"A tank fills in 6 hours with pipe A and 9 hours with pipe B. How many hours with both pipes open?",
"What is 15% of 15% of 4,000?",
"A price rises by 20% and then falls by 20%. What is the net change in percent?",
"How many positive divisors does 360 have?",
"A cyclist rides 18 km at 12 km/h and 18 km at 18 km/h. What is the average speed in km/h?",
"The ages of a mother and son add to 50. In 5 years the mother will be 3 times the son's age. How old is the son now?",
"What is the remainder when 2^20 is divided by 7?",
"A box has 4 red, 5 blue and 6 green balls. What is the probability that one ball drawn at random is not blue? Give a fraction.",
"Solve for x: 3(x - 4) + 7 = 2x + 9.",
"A recipe needs 250 g of flour for 6 cookies. How many grams for 27 cookies?",
"How many ways can 5 different books be arranged on a shelf?",
"The mean of 6 numbers is 14. One number is removed and the mean becomes 13. What number was removed?",
"A square has a diagonal of 10 cm. What is its area?",
"What is the smallest positive integer divisible by 2, 3, 4, 5 and 6?",
"A car uses 6 litres per 100 km. Fuel costs 105 rupees per litre. What does a 350 km trip cost?",
"Find the next number: 2, 6, 12, 20, 30, ?",
"A clock shows 3:15. What is the angle between the hour and minute hands in degrees?",
"How many integers from 1 to 100 are divisible by 3 or 5?",
"A worker is paid 600 rupees a day and fined 150 for each day absent. After 30 days she gets 13,500. How many days was she absent?",
"What is the sum of the interior angles of a hexagon in degrees?",
"If log10(x) = 3, what is x?",
"A coin is tossed 3 times. What is the probability of exactly 2 heads? Give a fraction.",
"Simplify: (2/3 + 3/4) divided by 17/12.",
"A ladder 13 m long reaches a window 12 m high. How far is the foot of the ladder from the wall in metres?",
"What is 999 x 101?",
"Two numbers have a sum of 40 and a difference of 8. What is their product?"
]


In [ ]:
%%writefile diag.py
#!/usr/bin/env python3
"""Diagnose the overthinker: where does the decision to STOP thinking live? (L02 hands-on, part 2)

  stopsignal  how likely the model is to close its thinking (</think>) at each point of saved answers
  easy        run easy questions (they finish), to get examples of the model deciding to stop
  finishdir   'finish' direction at every layer: residual just before </think> in finished answers
              minus residual late in looping answers
  steer       test a layer's finish direction at runtime (a hook, no weight change) on a few hard questions
  bake        write the best layer's finish direction into the weights (boost it in that layer's writers)

Small runs: --n 8 and --max-tokens 4096 by default, so each test takes minutes.
"""
import argparse, json, pathlib, re, time
import torch
from overthinker import load, layers_of, prompts, writers, RUNS, HERE


BOX = r"\\boxed\{[^{}]*\d[^{}]*\}"


def end_id(tok):
    i = tok.convert_tokens_to_ids("</think>")
    if i is None or i == tok.unk_token_id:
        raise SystemExit("This tokenizer has no </think> token.")
    return i


def gen(tok, model, ps, max_tokens, batch, stop_id):
    rows = []
    for i in range(0, len(ps), batch):
        enc = tok(ps[i:i + batch], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_tokens, do_sample=True, temperature=0.6, top_p=0.95, top_k=20,
                                 pad_token_id=tok.pad_token_id)
        for j, seq in enumerate(out):
            new = seq[enc["input_ids"].shape[1]:]
            closed = bool((new == stop_id).any())
            ended = bool((new == tok.eos_token_id).any() or (new == tok.pad_token_id).any())
            text = tok.decode(new, skip_special_tokens=False).replace(tok.pad_token or "\x00", "")
            rows.append({"prompt": ps[i + j], "output": text, "closed_thinking": closed, "ran_out": not ended})
    return rows


def summary(rows, label):
    def rep(o):
        w = o.split(); g = [tuple(w[k:k + 6]) for k in range(len(w) - 6)]
        return 1 - len(set(g)) / max(1, len(g))
    n = len(rows)
    boxed = sum(bool(re.search(BOX, r["output"])) for r in rows)
    reps = sum(rep(r["output"]) > 0.2 for r in rows)
    print(f"{label}: closed thinking {sum(r['closed_thinking'] for r in rows)}/{n} · ran out {sum(r['ran_out'] for r in rows)}/{n} · "
          f"boxed {boxed}/{n} · repeating {reps}/{n}")


def all_layers(model, enc):
    """Residual stream after every layer, in one forward pass (no vocabulary logits)."""
    got, hooks = {}, []
    for L, layer in enumerate(layers_of(model)):
        hooks.append(layer.register_forward_hook(lambda m, i, o, L=L: got.__setitem__(L, (o[0] if isinstance(o, tuple) else o).detach()[0])))
    try:
        with torch.no_grad():
            try: model(**enc, logits_to_keep=1)
            except TypeError: model(**enc)
    finally:
        for h in hooks: h.remove()
    return got


def stopsignal(a):
    """Probability of </think> (closing the thinking) at 40 evenly spaced points of each saved answer."""
    tok, model = load(a.model); sid = end_id(tok)
    rows = json.load(open(a.run))[:a.n]
    head = model.get_output_embeddings()
    norm = None
    for path in ("model.norm", "model.language_model.norm"):
        obj = model
        try:
            for part in path.split("."): obj = getattr(obj, part)
            norm = obj; break
        except AttributeError: pass
    bins = [[] for _ in range(10)]
    for r in rows:
        p_len = tok(r["prompt"], return_tensors="pt")["input_ids"].shape[1]
        enc = tok(r["prompt"] + r["output"], return_tensors="pt").to(model.device)
        got = {}
        h = norm.register_forward_hook(lambda m, i, o: got.__setitem__("h", o.detach()[0]))
        with torch.no_grad():
            try: model(**enc, logits_to_keep=1)
            except TypeError: model(**enc)
            h.remove()
            hid = got["h"][p_len:]
            idx = torch.linspace(0, len(hid) - 1, 40).long()
            prob = torch.softmax(head(hid[idx]).float(), -1)[:, sid].cpu()
        for k, pval in enumerate(prob.tolist()): bins[k * 10 // 40].append(pval)
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"chance of closing the thinking (</think>), start -> end of the answer, mean of {len(rows)} answers:")
    print("  " + "  ".join(f"{100 * sum(b) / len(b):5.2f}%" for b in bins))


def easy(a):
    tok, model = load(a.model); sid = end_id(tok)
    rows = gen(tok, model, prompts(tok, a.n, "easy.json"), a.max_tokens, a.batch, sid)
    json.dump(rows, open(RUNS / "observe-easy.json", "w"), indent=1)
    summary(rows, "easy questions")


def finishdir(a):
    tok, model = load(a.model); sid = end_id(tok)
    easy_rows = [r for r in json.load(open(RUNS / "observe-easy.json")) if r["closed_thinking"]]
    hard_rows = [r for r in json.load(open(a.hard)) if r["ran_out"]][:a.n]
    if not easy_rows: raise SystemExit("No easy answer closed its thinking. Run: python diag.py easy --model ...")
    pos, neg = {}, {}
    for r in easy_rows:
        text = r["output"]; cut = text.index("</think>")
        enc = tok(r["prompt"] + text[:cut], return_tensors="pt").to(model.device)
        for L, h in all_layers(model, enc).items(): pos.setdefault(L, []).append(h[-16:].mean(0).float().cpu())   # just before it stops
    for r in hard_rows:
        enc = tok(r["prompt"] + r["output"], return_tensors="pt").to(model.device)
        for L, h in all_layers(model, enc).items(): neg.setdefault(L, []).append(h[-256:].mean(0).float().cpu())  # deep in the loop
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    dirs, scale = {}, {}
    for L in pos:
        d = torch.stack(pos[L]).mean(0) - torch.stack(neg[L]).mean(0)
        dirs[L] = d / d.norm(); scale[L] = torch.stack(neg[L]).norm(dim=1).median().item()
    torch.save({"dirs": dirs, "scale": scale}, RUNS / "finish.pt")
    print(f"finish directions for {len(dirs)} layers, from {len(easy_rows)} finished vs {len(hard_rows)} looping answers")
    print("saved runs/finish.pt")


def steer(a):
    tok, model = load(a.model); sid = end_id(tok)
    f = torch.load(RUNS / "finish.pt")
    rows_all = []
    for L in a.layer:
        u = f["dirs"][L].to(model.device); s = f["scale"][L] * a.alpha
        hook = layers_of(model)[L].register_forward_hook(
            lambda m, i, o: ((o[0] + s * u,) + tuple(o[1:])) if isinstance(o, tuple) else o + s * u)
        t0 = time.time()
        rows = gen(tok, model, prompts(tok, a.n), a.max_tokens, a.batch, sid)
        hook.remove()
        summary(rows, f"steer layer {L} · alpha {a.alpha} · {time.time() - t0:.0f}s")
        for r in rows: r["layer"] = L
        rows_all += rows
    json.dump(rows_all, open(RUNS / f"steer-a{a.alpha}.json", "w"), indent=1)


def bake(a):
    """Weight edit: in layer L's writers, scale up the part of each column that lies along the finish direction.
    W' = W + beta * u (u^T W). Then save the model, and check it with: python overthinker.py observe --model ./qwen-finish"""
    tok, model = load(a.model)
    f = torch.load(RUNS / "finish.pt"); u = f["dirs"][a.layer].to(model.device)
    total = 0
    for name, lin in writers(layers_of(model)[a.layer], u.numel(), "all"):
        W = lin.weight.data
        W += a.beta * torch.outer(u, u @ W)
        total += W.numel()
        print(f"layer {a.layer} {name}: boosted the finish direction in {W.shape[1]} columns")
    out = HERE / a.out; model.save_pretrained(out); tok.save_pretrained(out)
    print(f"edited {total:,} weights · saved {out}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    x = sp.add_parser("stopsignal"); x.add_argument("--model", required=True); x.add_argument("--run", required=True); x.add_argument("--n", type=int, default=8)
    x = sp.add_parser("easy"); x.add_argument("--model", required=True); x.add_argument("--n", type=int, default=16)
    x.add_argument("--max-tokens", type=int, default=2048); x.add_argument("--batch", type=int, default=16)
    x = sp.add_parser("finishdir"); x.add_argument("--model", required=True); x.add_argument("--hard", default=str(RUNS / "observe-qwen3.5-0.8b.json"))
    x.add_argument("--n", type=int, default=8)
    x = sp.add_parser("steer"); x.add_argument("--model", required=True); x.add_argument("--layer", type=int, nargs="+", default=[15])
    x.add_argument("--alpha", type=float, default=0.5); x.add_argument("--n", type=int, default=8)
    x.add_argument("--max-tokens", type=int, default=4096); x.add_argument("--batch", type=int, default=8)
    x = sp.add_parser("bake"); x.add_argument("--model", required=True); x.add_argument("--layer", type=int, required=True)
    x.add_argument("--beta", type=float, default=1.0); x.add_argument("--out", default="qwen-finish")
    a = ap.parse_args()
    {"stopsignal": stopsignal, "easy": easy, "finishdir": finishdir, "steer": steer, "bake": bake}[a.cmd](a)


In [ ]:
%%writefile easy.json
["What is 7 + 5?","What is 9 x 6?","What is 100 - 37?","What is half of 84?","What is 3 squared?","What is 15 divided by 3?","How many minutes are in 2 hours?","What is 10% of 50?","What is 8 + 8 + 8?","What is 12 x 2?","What is 81 divided by 9?","What is 6 + 7?","How many days are in 3 weeks?","What is 1000 - 1?","What is 25 + 25?","What is 4 x 4?"]


## Download the model (about 1.5–2 GB)

In [ ]:
!pip install -q modelscope
from modelscope import snapshot_download
snapshot_download('Qwen/Qwen3.5-0.8B', local_dir='./qwen3.5-0.8b')
import os; print(sorted(os.listdir('./qwen3.5-0.8b')))

## Step 1 · Catch the overthinker (the 'before' run)
32 questions, up to 8,192 tokens each, on the T4.

In [ ]:
!python overthinker.py observe --model ./qwen3.5-0.8b --max-tokens 8192 --batch 16

## Step 2 · Find the stuck direction
Compares each answer before it starts repeating with its end, at layer 15.

In [ ]:
!python overthinker.py direction --model ./qwen3.5-0.8b --layer 15 --contrast early-late

## Step 3 · The last try: remove the direction from every layer
Every matrix that writes into the residual stream (attention output and MLP output), in all 24 layers.

In [ ]:
!python overthinker.py patch --model ./qwen3.5-0.8b --layer all --writers all --top 0

## Step 4 · Re-measure
Quick check on 8 questions first. If it clearly improves (fewer than 8 of 8 run out, or more boxed answers), run the next cell for all 32.

In [ ]:
!python overthinker.py observe --model ./qwen-patched --max-tokens 8192 --n 8 --batch 8

In [ ]:
!python overthinker.py observe --model ./qwen-patched --max-tokens 8192 --batch 16

## Look at one answer
Check that the edited model still writes sensible text (an edit this wide can also break it).

In [ ]:
import json
r = json.load(open('runs/observe-qwen-patched.json'))
print(r[0]['output'][:1500])
print('...')
print(r[0]['output'][-800:])

# Part 2 · Diagnose: where does the decision to stop live?
Small runs (8 questions, 4,096 tokens). About 1.5 hours in total.

**D1 · Does it ever want to stop?** Chance of closing the thinking, from the start to the end of the saved 'before' answers.

In [ ]:
!python diag.py stopsignal --model ./qwen3.5-0.8b --run runs/observe-qwen3.5-0.8b.json --n 8

**D2 · Easy questions it does finish**, to see what 'deciding to stop' looks like inside.

In [ ]:
!python diag.py easy --model ./qwen3.5-0.8b

**D3 · A 'finish' direction at every layer**: just before it stops (easy answers) minus deep in a loop (hard answers).

In [ ]:
!python diag.py finishdir --model ./qwen3.5-0.8b

**Baseline for the small runs** (8 questions, 4,096 tokens, no change to the model). Compare every D4–D6 result with this.

In [ ]:
# baseline at the same small settings (alpha 0 = no push)
!python diag.py steer --model ./qwen3.5-0.8b --layer 6 --alpha 0

**D4 · Layer scan.** Push the finish direction in at runtime (a hook, no weight change), one layer at a time. About 6–7 minutes per layer. Look for 'closed thinking' going up.

In [ ]:
!python diag.py steer --model ./qwen3.5-0.8b --layer 6 10 14 18 22 --alpha 0.5

**D5 · Tune the best layer.** Replace 14 with the best layer from D4.

In [ ]:
!python diag.py steer --model ./qwen3.5-0.8b --layer 14 --alpha 0.25
!python diag.py steer --model ./qwen3.5-0.8b --layer 14 --alpha 1.0

**D6 · Write it into the weights, and check.** Replace 14 with the best layer. Beta sets how strongly the direction is boosted.

In [ ]:
!python diag.py bake --model ./qwen3.5-0.8b --layer 14 --beta 1.0
!python overthinker.py observe --model ./qwen-finish --max-tokens 4096 --n 8 --batch 8